In [0]:
dbutils.widgets.text("ADZUNA_APP_ID", "")
dbutils.widgets.text("ADZUNA_APP_KEY", "")

In [0]:
APP_ID = dbutils.widgets.get("ADZUNA_APP_ID")
APP_KEY = dbutils.widgets.get("ADZUNA_APP_KEY")

if not APP_ID or not APP_KEY:
    raise ValueError("Missing Adzuna credentials")

In [0]:
search_terms = [

    # Data Engineering
    "data engineer",
    "data architect",

    # Analytics
    "data analyst",
    "business intelligence analyst",

    # Data Science
    "data scientist",

    # Machine Learning
    "machine learning engineer",

    # AI
    "ai engineer",

    # MLOps
    "mlops engineer",

    # Database
    "database engineer",

    # Emerging
    "analytics engineer"
]

In [0]:
country_codes = [
    "us",
    "gb",
    "ca",
    "au"
]

In [0]:
import requests
from pyspark.sql.functions import *

all_jobs = []

for country in country_codes:

    for term in search_terms:

        print(f"Fetching {country} | {term}")

        url = (
            f"https://api.adzuna.com/v1/api/jobs/{country}/search/1"
            f"?app_id={APP_ID}"
            f"&app_key={APP_KEY}"
            f"&results_per_page=50"
            f"&what={term}"
        )

        response = requests.get(url)

        if response.status_code == 200:

            data = response.json()

            if "results" in data:
                all_jobs.extend(data["results"])

Fetching us | data engineer
Fetching us | data architect
Fetching us | data analyst
Fetching us | business intelligence analyst
Fetching us | data scientist
Fetching us | machine learning engineer
Fetching us | ai engineer
Fetching us | mlops engineer
Fetching us | database engineer
Fetching us | analytics engineer
Fetching gb | data engineer
Fetching gb | data architect
Fetching gb | data analyst
Fetching gb | business intelligence analyst
Fetching gb | data scientist
Fetching gb | machine learning engineer
Fetching gb | ai engineer
Fetching gb | mlops engineer
Fetching gb | database engineer
Fetching gb | analytics engineer
Fetching ca | data engineer
Fetching ca | data architect
Fetching ca | data analyst
Fetching ca | business intelligence analyst
Fetching ca | data scientist
Fetching ca | machine learning engineer
Fetching ca | ai engineer
Fetching ca | mlops engineer
Fetching ca | database engineer
Fetching ca | analytics engineer
Fetching au | data engineer
Fetching au | data ar

In [0]:
print("Raw Jobs:", len(all_jobs))

if len(all_jobs) == 0:
    raise Exception("No jobs returned from Adzuna API")

Raw Jobs: 1959


In [0]:
unique_jobs = {}

for job in all_jobs:
    unique_jobs[job["id"]] = job

all_jobs = list(unique_jobs.values())

print("Unique Jobs:", len(all_jobs))

Unique Jobs: 1941


In [0]:
from pyspark.sql import Row

jobs = []

for job in all_jobs:

    jobs.append(
        Row(
            job_id=str(job.get("id")),

            job_title=str(job.get("title"))
            if job.get("title") else None,

            company=str(job.get("company", {}).get("display_name"))
            if job.get("company") else None,

            location=str(job.get("location", {}).get("display_name"))
            if job.get("location") else None,

            country=str(job.get("location", {}).get("area", [None])[0])
            if job.get("location") else None,

            category=str(job.get("category", {}).get("label"))
            if job.get("category") else None,

            description=str(job.get("description"))
            if job.get("description") else None,

            salary_min=float(job.get("salary_min"))
            if job.get("salary_min") is not None
            else None,

            salary_max=float(job.get("salary_max"))
            if job.get("salary_max") is not None
            else None,

            contract_time=str(job.get("contract_time"))
            if job.get("contract_time") else None,

            created_date=str(job.get("created"))
            if job.get("created") else None,

            job_url=str(job.get("redirect_url"))
            if job.get("redirect_url") else None
        )
    )

df_bronze = spark.createDataFrame(jobs)

print("Rows:", df_bronze.count())

Rows: 1941


In [0]:
from pyspark.sql.functions import *

exclude_patterns = (
    "data center|datacenter|construction|controls engineer|"
    "mechanical engineer|facility engineer|civil engineer"
)

df_bronze = (
    df_bronze.filter(
        ~lower(col("job_title")).rlike(exclude_patterns)
    )
)

print("Filtered Count:", df_bronze.count())

Filtered Count: 1931


In [0]:
df_bronze = (
    df_bronze
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("ingestion_date", current_date())
    .withColumn("source_system", lit("ADZUNA"))
)

In [0]:
allowed_categories = [
    "IT Jobs",
    "Engineering Jobs",
    "Scientific & QA Jobs",
    "Consultancy Jobs"
]

df_bronze = df_bronze.filter(
    col("category").isin(allowed_categories)
)

In [0]:
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("skillpulse.bronze.bronze_jobs_realtime")
)

In [0]:
print(
    "Bronze Jobs Saved:",
    df_bronze.count()
)

Bronze Jobs Saved: 1871
